# MVP - Data Engineering

## Camada silver

Nesta camada, limparemos os dados, para evitar problemas na fase de análise.

Primeiro, faremos um diagnóstico de dados para saber quais são os problemas: se há valores nulos, duplicados ou inconsistência de dados.

Depois, faremos os ajustes necessários, removendo colunas que não farão parte do modelo final, eliminando linhas duplicadas (quando aplicável) e transformaremos valores ausentes. Também verificaremos a consistência de dados em algumas colunas.

Daí, estaremos prontos para criar as tabelas da camada silver.
Este notebook também utiliza as funções definidas do notebook common.

In [0]:
%run /Users/cyntia_invernizzi@hotmail.com/_DataEng_PUCRIO/common

In [0]:
# Python Libraries

from pyspark.sql.functions import col, when, isnull, sum as spark_sum, count as count_all, expr
from pyspark.sql.types import StringType, IntegerType, DoubleType, TimestampType, DateType
from pyspark.sql.window import Window


In [0]:
%sql
-- Estabelecendo o catálogo usado no MVP (SQL)

USE CATALOG mvp_pucrio;

In [0]:
# Estabelecendo as variáveis para catálogo e schema para serem usadas neste Notebook em Pyspark.

catalog = "mvp_pucrio"

## Diagnóstico de Dados

Antes de montar a camada silver, verificamos se há dados nulos e duplicados. Também verificamos se os estados e os despachos contém os valores esperados. Assim, saberemos que transformações são necessárias antes de obtermos o modelo final.

In [0]:
def check_state(df, table_name):
    """Verifica se a coluna de estado contém apenas UFs brasileiras válidas."""
    brazilian_states = ["AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO", "unknown"]

    if table_name in ("customers", "fato_vendas"):
        state = "customer_state"
    elif table_name in ("sellers",):
        state = "seller_state"
    else:
        state = "state"

    wrong_state = df.filter(~df[state].isin(brazilian_states))

    problems = 0
    if wrong_state.count() > 0:
        problems = 1
        print(f"\nQuantidade de linhas com estado inválido: {wrong_state.count()}")
        wrong_state.show(truncate=False)

    return problems

In [0]:
def check_shipment(df, table_name):
    """Verifica se order_status contém apenas valores válidos."""
    valid_status = ["delivered", "shipped", "canceled", "processing", "unavailable", "invoiced", "created", "approved"]

    invalid_status = df.filter(~col("order_status").isin(valid_status))
    count = invalid_status.count()

    problems = 0
    if count > 0:
        problems = 1
        print(f"\nQuantidade de linhas com order_status desconhecido: {count}")
        invalid_status.select("order_status").distinct().show(truncate=False)

    return problems

In [0]:
# Encontrando duplicados e nulos na camada bronze para serem ajustados nesta camada.
 
nulls_dict = dict()
duplicates_set = set()

print(f"Diagnóstico de Dados - Tabelas que requerem limpeza (bronze):")
problems = 0

schema = f"{catalog}.bronze"
table_names = schema_tables(schema)

for name in table_names:
    full_name = f"{schema}.{name}"
    df = spark.table(full_name)

    num_nulls, problems, nulls_dict = find_nulls(df, name, problems, nulls_dict)
    
    problems, duplicates_set = find_duplicates(df, "silver", name, num_nulls, problems, duplicates_set)

    if name in ("sellers", "customers"):
        problems += check_state(df, name)

    if name == "orders":
        problems += check_shipment(df, name)

if problems == 0:
    print("Nenhuma tabela apresenta valores nulos ou duplicados.")

Os resultados acima indicam algumas ações que devem ser tomadas:

1. Colunas com valores nulos podem ser alterados para "unknown".
2. Duplicações na planilha "order_items" desaparecerão, já que as linhas serão agrupadas, somando os valores da coluna price (renomeada para sales_value) e calculando a quantidade de itens (quantity).

### Limpeza dos dados

Nesta seção, limparemos os dados, para evitar problemas na fase de análise. Primeiro, removeremos colunas que não farão parte do modelo final. Depois, conforme identificado na seção de "Diagnóstico de dados", agruparemos tabelas que contém valores duplicados e transformaremos valores ausentes para "unknown".

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS silver;

Abaixo estou definido a tipagem de cada tabela da camada silver.

In [0]:
# Dicionário define o tipo esperado das colunas de cada tabela na camada Silver

silver_schema = {
    "customers": {
        "customer_id": StringType(),
        "customer_city": StringType(),
        "customer_state": StringType(),
    },
    "sellers": {
        "seller_id": StringType(),
        "seller_city": StringType(),
        "seller_state": StringType(),
    },
    "marketing_qualified_leads": {
        "mql_id": StringType(),
        "first_contact_date": DateType(),
        "origin": StringType(),
    },
    "closed_deals": {
        "mql_id": StringType(),
        "seller_id": StringType(),
        "won_date": DateType()
    },
    "products": {
        "product_id": StringType(),
        "product_category_name": StringType(),
    },
    "orders": {
        "order_id": StringType(),
        "customer_id": StringType(),
        "order_status": StringType(),
        "order_date": DateType(),
    },
    "order_items": {
        "order_id": StringType(),
        "product_id": StringType(),
        "seller_id": StringType(),
        "quantity": IntegerType(),
        "sales_value": DoubleType(),
    },
}

In [0]:
def apply_schema(df, table_name):
    """Aplica os tipos esperados às colunas do DataFrame conforme o schema da silver."""
    schema = silver_schema.get(table_name, {})
    for col_name, col_type in schema.items():
        if col_name in df.columns:
            df = df.withColumn(col_name, col(col_name).cast(col_type))
    return df

In [0]:
# Removendo colunas

def removing_cols(df):
    """Remove colunas que não serão usadas no modelo"""
    cols_to_remove = set(df.columns) - get_columns_to_model()
    return df.drop(*cols_to_remove)

Abaixo, para cada tabela bronze:
- as colunas que não fazem parte do modelo são removidas;
- items duplicados na tabela "order_items" são agrupados por order_id, product_id e seller_id, contando a quantidade de produtos na ordem (COUNT de linhas por produto) e somando os valores de prices (renomeado para sales_value);
- valores ausentes são preenchidos com "unknown";
- a tipagem de cada tabela é aplicada.

Finalmente, as tabelas são "escritas" na camada silver.

In [0]:
# Aplicando funções às planilhas

for table in table_names:
    df = spark.table(f"{catalog}.bronze.{table}")
    
    if table == "orders":
        df = df.withColumn("order_date", col("order_purchase_timestamp").cast("date"))
    
    df = removing_cols(df)
    
    if table == "order_items":
        df = df.groupBy(["order_id", "product_id", "seller_id"]).agg(
            count_all("*").alias("quantity"),
            spark_sum("price").alias("sales_value")
        )

    if table in nulls_dict.keys():
        df = df.fillna("unknown", subset=nulls_dict[table])
    
    df = apply_schema(df, table)
    
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.{table}")

In [0]:
%sql
-- Configurando contexto para documentação

USE SCHEMA silver;

Como agrupamos alguns dados na tabela order_items, precisamos verificar se a soma de sales_value continua batendo com o total de "price" que tinhamos na tabela bronze.

In [0]:
%sql
SELECT ROUND(
  (SELECT SUM(sales_value) FROM order_items) - (SELECT SUM(price) FROM bronze.order_items)
, 2) AS checksum;

### Documentando as tabelas

Abaixo somente adicionando a documentação a cada tabela.

In [0]:
%sql
-- Documentando tabela customers

COMMENT ON TABLE customers IS 
'Dimensão de clientes. Origem: bronze.customers. Contém apenas colunas do modelo: customer_id, customer_city, customer_state.';

COMMENT ON COLUMN customers.customer_id IS 
'Primary Key - ID único do cliente. Tipo: STRING.';

COMMENT ON COLUMN customers.customer_city IS 
'Cidade do cliente. Tipo: STRING. Valores: cidades brasileiras ou "unknown".';

COMMENT ON COLUMN customers.customer_state IS 
'Estado (UF) do cliente. Tipo: STRING. Valores: SP, RJ, MG, etc ou "unknown".';

In [0]:
%sql
-- Documentando tabela sellers

COMMENT ON TABLE sellers IS 
'Dimensão de sellers (vendedores). Origem: bronze.sellers. Contém apenas colunas do modelo: seller_id, seller_city, seller_state.';

COMMENT ON COLUMN sellers.seller_id IS 
'Primary Key - ID único do seller. Tipo: STRING.';

COMMENT ON COLUMN sellers.seller_city IS 
'Cidade do seller. Tipo: STRING. Valores: cidades brasileiras ou "unknown".';

COMMENT ON COLUMN sellers.seller_state IS 
'Estado (UF) do seller. Tipo: STRING. Valores: SP, RJ, MG, etc ou "unknown".';

In [0]:
%sql
-- Documentando tabela marketing_qualified_leads

COMMENT ON TABLE marketing_qualified_leads IS 
'Leads qualificados de marketing (MQLs). Origem: bronze.marketing_qualified_leads. Contém apenas colunas do modelo: mql_id, first_contact_date, origin.';

COMMENT ON COLUMN marketing_qualified_leads.mql_id IS 
'Primary Key - ID único do lead. Tipo: STRING.';

COMMENT ON COLUMN marketing_qualified_leads.first_contact_date IS 
'Data do primeiro contato com o lead. Tipo: DATE. Formato: YYYY-MM-DD.';

COMMENT ON COLUMN marketing_qualified_leads.origin IS 
'Canal de origem do lead (organic_search, paid_search, social, email, direct, etc ou "unknown"). Tipo: STRING.';

In [0]:
%sql
-- Documentando tabela closed_deals

COMMENT ON TABLE closed_deals IS 
'Negócios fechados (leads convertidos em sellers). Origem: bronze.closed_deals. Contém apenas colunas do modelo: mql_id, seller_id, won_date.';

COMMENT ON COLUMN closed_deals.mql_id IS 
'Foreign Key para marketing_qualified_leads.mql_id. ID do lead que foi convertido. Tipo: STRING.';

COMMENT ON COLUMN closed_deals.seller_id IS 
'ID do seller resultante da conversão do lead. Tipo: STRING.';

COMMENT ON COLUMN closed_deals.won_date IS 
'Data em que o negócio foi fechado. Tipo: DATE. Formato: YYYY-MM-DD.';

In [0]:
%sql
-- Documentando tabela products

COMMENT ON TABLE products IS 
'Dimensão de produtos. Origem: bronze.products. Contém apenas colunas do modelo: product_id, product_category_name.';

COMMENT ON COLUMN products.product_id IS 
'Primary Key - ID único do produto. Tipo: STRING.';

COMMENT ON COLUMN products.product_category_name IS 
'Nome da categoria do produto (ex: cama_mesa_banho, beleza_saude, eletronicos, etc ou "unknown"). Tipo: STRING.';

In [0]:
%sql
-- Documentando tabela orders

COMMENT ON TABLE orders IS 
'Tabela de pedidos. Origem: bronze.orders. Contém apenas colunas do modelo: order_id, customer_id, order_status, order_date.';

COMMENT ON COLUMN orders.order_id IS 
'Primary Key - ID único do pedido. Tipo: STRING.';

COMMENT ON COLUMN orders.customer_id IS 
'Foreign Key para customers.customer_id. ID do cliente que fez o pedido. Tipo: STRING.';

COMMENT ON COLUMN orders.order_status IS 
'Status do pedido (delivered, shipped, canceled, processing, invoiced, approved, created, unavailable). Tipo: STRING.';

COMMENT ON COLUMN orders.order_date IS 
'Data da compra. Tipo: DATE. Formato: YYYY-MM-DD. Derivada de bronze.orders.order_purchase_timestamp via CAST.';

In [0]:
%sql
-- Documentando tabela order_items

COMMENT ON TABLE order_items IS 
'Itens de pedido agregados por order_id + product_id + seller_id. Origem: bronze.order_items (agregação). Contém: order_id, product_id, seller_id, quantity, sales_value.';

COMMENT ON COLUMN order_items.order_id IS 
'Foreign Key para orders.order_id. ID do pedido. Tipo: STRING. Parte da PK composta.';

COMMENT ON COLUMN order_items.product_id IS 
'Foreign Key para products.product_id. ID do produto vendido. Tipo: STRING. Parte da PK composta.';

COMMENT ON COLUMN order_items.seller_id IS 
'Foreign Key para sellers.seller_id. ID do seller que vendeu. Tipo: STRING. Parte da PK composta.';

COMMENT ON COLUMN order_items.quantity IS 
'Quantidade de itens do mesmo produto no pedido. Tipo: INT. Valores: >= 1. Agregável: SUM, AVG.';

COMMENT ON COLUMN order_items.sales_value IS 
'Valor total vendido (soma de price). Tipo: DOUBLE. Valores: >= 0. Agregável: SUM, AVG.';

### Verificando limpeza das tabelas

Para nos assegurarmos que as limpeza dos dados foi concluída com sucesso, verificamos novamente se há duplicados ou valores ausentes.

In [0]:
# Verificando limpeza de dados

nulls_dict = dict()
duplicates_set = set()

print(f"Diagnóstico de Dados - Tabelas que requerem limpeza (silver):")
problems = 0

schema = f"{catalog}.silver"
table_names = schema_tables(schema)

for name in table_names:
    full_name = f"{schema}.{name}"
    df = spark.table(full_name)

    num_nulls, problems, nulls_dict = find_nulls(df, name, problems, nulls_dict)
    
    problems, duplicates_set = find_duplicates(df, "silver", name, num_nulls, problems, duplicates_set)

if problems == 0:
    print("Nenhuma tabela apresenta valores nulos ou duplicados.")

Conforme mostrado acima, os nossos dados estão limpos!
Abaixo, uma rápida visualização de cada tabela.

In [0]:
%sql
-- Verificando customers

SELECT *
FROM customers
ORDER BY customer_id
LIMIT 10

In [0]:
%sql
-- Verificando sellers

SELECT *
FROM sellers
ORDER BY seller_id
LIMIT 10

In [0]:
%sql
-- Verificando marketing_qualified_leads

SELECT *
FROM marketing_qualified_leads
ORDER BY mql_id
LIMIT 10

In [0]:
%sql
-- Verificando closed_deals

SELECT *
FROM closed_deals
ORDER BY mql_id
LIMIT 10

In [0]:
%sql
-- Verificando products

SELECT *
FROM products
ORDER BY product_id
LIMIT 10

In [0]:
%sql
-- Verificando orders

SELECT *
FROM orders
ORDER BY order_id
LIMIT 10

In [0]:
%sql
-- Verificando order_items

SELECT *
FROM order_items
ORDER BY order_id, product_id, seller_id
LIMIT 10